# Cohorte Sepsis-3 — MIMIC-IV 3.0

Construcción paso a paso de la cohorte con criterios Sepsis-3:
- **Infección sospechada**: criterios de Angus (cultivo + antibiótico IV ±24h)
- **Disfunción orgánica**: SOFA ≥ 2 en ventana [−48h, +24h] del onset de infección

Salida: `data/processed/cohort.parquet`

In [1]:
import polars as pl
import numpy as np
import json
from pathlib import Path
import time

MIMIC   = Path.home() / "mimic-iv-3.0"
HOSP    = MIMIC / "hosp"
ICU     = MIMIC / "icu"
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

ANTIBIOTICS = [
    "vancomycin", "piperacillin", "tazobactam", "meropenem", "imipenem",
    "cefepime", "ceftriaxone", "ceftazidime", "ciprofloxacin", "levofloxacin",
    "metronidazole", "ampicillin", "sulbactam", "gentamicin", "tobramycin",
    "amikacin", "aztreonam", "linezolid", "daptomycin", "colistin",
    "tigecycline", "cefazolin", "nafcillin", "oxacillin", "clindamycin",
    "trimethoprim", "sulfamethoxazole", "fluconazole", "micafungin",
    "caspofungin", "anidulafungin", "voriconazole", "amphotericin",
    "doxycycline", "azithromycin", "erythromycin", "rifampin", "ertapenem",
    "doripenem", "cefuroxime", "cefoxitin", "cefotetan", "ticarcillin",
]

LAB_ITEMS = {
    "platelet":   [51265],
    "bilirubin":  [50885],
    "creatinine": [50912],
    "lactate":    [50813, 52442],
    "wbc":        [51301],
    "hemoglobin": [51222],
}

CHART_ITEMS = {
    "gcs_eye":    [220739],
    "gcs_verbal": [223900],
    "gcs_motor":  [223901],
    "map_ni":     [220181],
    "map_art":    [220052],
    "spo2":       [220277],
    "fio2":       [223835],
    "resp_rate":  [220210],
    "heart_rate": [220045],
}

VASOPRESSOR_ITEMS = [221906, 221289, 222315, 221662, 221653]

print("Config OK")

Config OK


## PASO 1 — Cohorte base (stays ≥24h)

In [2]:
stays_raw = (
    pl.read_csv(ICU / "icustays.csv.gz")
    .with_columns([
        pl.col("intime").str.to_datetime(),
        pl.col("outtime").str.to_datetime(),
        pl.col("hadm_id").cast(pl.Int64),
        pl.col("stay_id").cast(pl.Int64),
        pl.col("subject_id").cast(pl.Int64),
    ])
    .with_columns(
        ((pl.col("outtime") - pl.col("intime")).dt.total_hours()).alias("los_hours")
    )
)
print(f"Stays brutos (sin filtro): {len(stays_raw):,}")

stays = stays_raw.filter(pl.col("los_hours") >= 24)

print(f"Stays ≥24h: {len(stays):,}")
print(f"Excluidas (<24h): {len(stays_raw) - len(stays):,}")
print(f"Pacientes únicos: {stays['subject_id'].n_unique():,}")
stays.head(3)

Stays brutos (sin filtro): 94,458
Stays ≥24h: 74,829
Excluidas (<24h): 19,629
Pacientes únicos: 54,551


[Vista previa de registros individuales omitida — DUA de PhysioNet: los datos de MIMIC-IV no se redistribuyen]


## PASO 2 — Infección sospechada (criterios de Angus)

In [3]:
log("Cargando microbiologyevents ...")
cultures = (
    pl.read_csv(
        HOSP / "microbiologyevents.csv.gz",
        columns=["subject_id", "hadm_id", "charttime", "spec_type_desc"],
    )
    .with_columns([
        pl.col("charttime").str.to_datetime(strict=False),
        pl.col("hadm_id").cast(pl.Int64),
        pl.col("subject_id").cast(pl.Int64),
    ])
    .drop_nulls(subset=["charttime", "hadm_id"])
    .filter(
        pl.col("spec_type_desc").str.to_lowercase().str.contains(
            "blood|urine|sputum|bronchoalveolar|csf|peritoneal"
        )
    )
)

print(f"Cultivos relevantes: {len(cultures):,}")
print(cultures["spec_type_desc"].value_counts().sort("count", descending=True).head(8))

[19:28:06] Cargando microbiologyevents ...


Cultivos relevantes: 1,110,500
shape: (8, 2)
┌─────────────────────────────────┬────────┐
│ spec_type_desc                  ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u32    │
╞═════════════════════════════════╪════════╡
│ BLOOD CULTURE                   ┆ 424498 │
│ URINE                           ┆ 330221 │
│ SPUTUM                          ┆ 182028 │
│ BRONCHOALVEOLAR LAVAGE          ┆ 50824  │
│ PERITONEAL FLUID                ┆ 39573  │
│ CSF;SPINAL FLUID                ┆ 22554  │
│ SEROLOGY/BLOOD                  ┆ 18594  │
│ BLOOD CULTURE ( MYCO/F LYTIC B… ┆ 10206  │
└─────────────────────────────────┴────────┘


In [4]:
log("Cargando emar (antibióticos IV) ...")
emar = (
    pl.read_csv(
        HOSP / "emar.csv.gz",
        columns=["subject_id", "hadm_id", "medication", "scheduletime", "event_txt"],
    )
    .with_columns([
        pl.col("scheduletime").str.to_datetime(strict=False),
        pl.col("medication").str.to_lowercase(),
        pl.col("hadm_id").cast(pl.Int64),
        pl.col("subject_id").cast(pl.Int64),
    ])
    .drop_nulls(subset=["scheduletime", "hadm_id"])
)

abx_mask = pl.lit(False)
for abx in ANTIBIOTICS:
    abx_mask = abx_mask | pl.col("medication").str.contains(abx)
emar_abx = emar.filter(abx_mask)

print(f"Registros de antibióticos: {len(emar_abx):,}")
print(emar_abx["medication"].value_counts().sort("count", descending=True).head(10))

[19:28:09] Cargando emar (antibióticos IV) ...


Registros de antibióticos: 2,257,139
shape: (10, 2)
┌─────────────────────────┬────────┐
│ medication              ┆ count  │
│ ---                     ┆ ---    │
│ str                     ┆ u32    │
╞═════════════════════════╪════════╡
│ vancomycin              ┆ 316392 │
│ metronidazole           ┆ 257330 │
│ cefepime                ┆ 201974 │
│ ceftriaxone             ┆ 165881 │
│ piperacillin-tazobactam ┆ 156190 │
│ cefazolin               ┆ 150477 │
│ vancomycin oral liquid  ┆ 124969 │
│ meropenem               ┆ 109486 │
│ ampicillin-sulbactam    ┆ 74080  │
│ ciprofloxacin hcl       ┆ 74056  │
└─────────────────────────┴────────┘


In [5]:
log("Buscando coincidencias cultivo + antibiótico (±24h) ...")

cultures_slim = cultures.select(["subject_id", "hadm_id", "charttime"]).rename({"charttime": "culture_time"})
abx_slim = emar_abx.select(["hadm_id", "scheduletime"]).rename({"scheduletime": "abx_time"})

infection = (
    cultures_slim
    .join(abx_slim, on="hadm_id", how="inner")
    .with_columns(
        (pl.col("abx_time") - pl.col("culture_time")).dt.total_hours().alias("diff_hours")
    )
    .filter(pl.col("diff_hours").abs() <= 24)
    .with_columns(
        pl.when(pl.col("abx_time") < pl.col("culture_time"))
        .then(pl.col("abx_time"))
        .otherwise(pl.col("culture_time"))
        .alias("infection_time")
    )
    # FIX: group_by solo hadm_id — evita filas duplicadas cuando el mismo hadm_id
    # aparece con distintos subject_id en microbiologyevents (calidad de datos MIMIC)
    .group_by("hadm_id")
    .agg(pl.col("infection_time").min())
)

print(f"Admisiones con infección sospechada: {infection['hadm_id'].n_unique():,}")
print(f"Filas en infection (debe == n_unique): {len(infection):,}")
infection.head(5)

[19:28:39] Buscando coincidencias cultivo + antibiótico (±24h) ...


Admisiones con infección sospechada: 58,108
Filas en infection (debe == n_unique): 58,108


[Vista previa de registros individuales omitida — DUA de PhysioNet: los datos de MIMIC-IV no se redistribuyen]


## PASO 3 — Cargar datos SOFA

> **Fix aplicado**: cast explícito de `hadm_id` y `stay_id` a `Int64` al cargar, 
> evitando el error `cannot compare string with numeric type` en el filtro posterior.

In [6]:
log("Cargando labevents (puede tardar ~90s) ...")
all_lab_ids = [item for ids in LAB_ITEMS.values() for item in ids]

labs = (
    pl.read_csv(
        HOSP / "labevents.csv.gz",
        columns=["subject_id", "hadm_id", "itemid", "charttime", "valuenum"],
    )
    .with_columns([
        pl.col("charttime").str.to_datetime(strict=False),
        pl.col("itemid").cast(pl.Int64),
        pl.col("hadm_id").cast(pl.Int64),   # FIX: era String, causaba ComputeError
        pl.col("subject_id").cast(pl.Int64),
    ])
    .filter(
        pl.col("itemid").is_in(all_lab_ids) &
        pl.col("valuenum").is_not_null() &
        pl.col("hadm_id").is_not_null()
    )
)

print(f"Registros de laboratorio SOFA: {len(labs):,}")
print(f"Schema: {labs.schema}")
labs.head(3)

[19:28:39] Cargando labevents (puede tardar ~90s) ...


Registros de laboratorio SOFA: 11,125,123
Schema: Schema([('subject_id', Int64), ('hadm_id', Int64), ('itemid', Int64), ('charttime', Datetime(time_unit='us', time_zone=None)), ('valuenum', Float64)])


[Vista previa de registros individuales omitida — DUA de PhysioNet: los datos de MIMIC-IV no se redistribuyen]


In [7]:
log("Cargando chartevents SOFA (puede tardar ~3 min) ...")
all_chart_ids = [item for ids in CHART_ITEMS.values() for item in ids]

charts = (
    pl.read_csv(
        ICU / "chartevents.csv.gz",
        columns=["subject_id", "stay_id", "itemid", "charttime", "valuenum"],
    )
    .with_columns([
        pl.col("charttime").str.to_datetime(strict=False),
        pl.col("itemid").cast(pl.Int64),
        pl.col("stay_id").cast(pl.Int64),   # FIX: cast explícito
        pl.col("subject_id").cast(pl.Int64),
    ])
    .filter(
        pl.col("itemid").is_in(all_chart_ids) &
        pl.col("valuenum").is_not_null()
    )
)

print(f"Chartevents SOFA: {len(charts):,}")
print(f"Schema: {charts.schema}")

[19:29:56] Cargando chartevents SOFA (puede tardar ~3 min) ...


Chartevents SOFA: 42,184,134
Schema: Schema([('subject_id', Int64), ('stay_id', Int64), ('charttime', Datetime(time_unit='us', time_zone=None)), ('itemid', Int64), ('valuenum', Float64)])


In [8]:
log("Cargando vasopresores ...")
vasopress = (
    pl.read_csv(
        ICU / "inputevents.csv.gz",
        columns=["subject_id", "stay_id", "itemid", "starttime", "endtime", "rate"],
    )
    .with_columns([
        pl.col("starttime").str.to_datetime(strict=False),
        pl.col("endtime").str.to_datetime(strict=False),
        pl.col("itemid").cast(pl.Int64),
        pl.col("stay_id").cast(pl.Int64),   # FIX: cast explícito
    ])
    .filter(pl.col("itemid").is_in(VASOPRESSOR_ITEMS))
)

print(f"Registros vasopresores: {len(vasopress):,}")

[19:34:56] Cargando vasopresores ...


Registros vasopresores: 556,807


## PASO 4 — SOFA vectorizado

En vez de iterar fila a fila sobre 23k stays (lento, ~4h), 
calculamos el SOFA con joins y expresiones Polars (minutos).

In [9]:
# Unir stays con infection_time y calcular ventana
stays_inf = (
    stays.join(infection.select(["hadm_id", "infection_time"]), on="hadm_id", how="left")
)
stays_inf_only = stays_inf.filter(pl.col("infection_time").is_not_null())

print(f"Stays con infección sospechada: {len(stays_inf_only):,}")

# Añadir ventana temporal [-48h, +24h]
stays_win = stays_inf_only.with_columns([
    (pl.col("infection_time") - pl.duration(hours=48)).alias("t_start"),
    (pl.col("infection_time") + pl.duration(hours=24)).alias("t_end"),
])

Stays con infección sospechada: 23,095


In [10]:
# ── Labs SOFA vectorizado ────────────────────────────────────────
log("Calculando SOFA labs (vectorizado) ...")

labs_win = (
    stays_win.select(["hadm_id", "t_start", "t_end"])
    .join(labs, on="hadm_id", how="left")
    .filter(
        (pl.col("charttime") >= pl.col("t_start")) &
        (pl.col("charttime") <= pl.col("t_end"))
    )
)

sofa_labs_parts = []
for name, item_ids in LAB_ITEMS.items():
    part = (
        labs_win.filter(pl.col("itemid").is_in(item_ids))
        .group_by("hadm_id")
        .agg(pl.col("valuenum").mean().alias(name))
    )
    sofa_labs_parts.append(part)

# FIX: .unique() — varios stay_id pueden compartir hadm_id (misma admisión, varios ingresos UCI);
# sin unique(), sofa_labs tendría hadm_id duplicados → join muchos-a-muchos en sofa_total
sofa_labs = stays_win.select("hadm_id").unique()
for part in sofa_labs_parts:
    sofa_labs = sofa_labs.join(part, on="hadm_id", how="left")

sofa_labs = sofa_labs.with_columns([
    pl.when(pl.col("platelet").is_null()).then(0)
      .when(pl.col("platelet") > 150).then(0)
      .when(pl.col("platelet") > 100).then(1)
      .when(pl.col("platelet") > 50).then(2)
      .when(pl.col("platelet") > 20).then(3)
      .otherwise(4).cast(pl.Int8).alias("sofa_coag"),
    pl.when(pl.col("bilirubin").is_null()).then(0)
      .when(pl.col("bilirubin") < 1.2).then(0)
      .when(pl.col("bilirubin") < 2.0).then(1)
      .when(pl.col("bilirubin") < 6.0).then(2)
      .when(pl.col("bilirubin") < 12.0).then(3)
      .otherwise(4).cast(pl.Int8).alias("sofa_liver"),
    pl.when(pl.col("creatinine").is_null()).then(0)
      .when(pl.col("creatinine") < 1.2).then(0)
      .when(pl.col("creatinine") < 2.0).then(1)
      .when(pl.col("creatinine") < 3.5).then(2)
      .when(pl.col("creatinine") < 5.0).then(3)
      .otherwise(4).cast(pl.Int8).alias("sofa_renal"),
])

print(f"SOFA labs: {len(sofa_labs):,} hadm_ids únicos")
sofa_labs.select(["hadm_id", "platelet", "sofa_coag", "creatinine", "sofa_renal"]).head(5)

[19:35:05] Calculando SOFA labs (vectorizado) ...


SOFA labs: 19,777 hadm_ids únicos


[Vista previa de registros individuales omitida — DUA de PhysioNet: los datos de MIMIC-IV no se redistribuyen]


In [11]:
# ── Charts SOFA vectorizado ──────────────────────────────────────
log("Calculando SOFA charts (vectorizado) ...")

charts_win = (
    stays_win.select(["stay_id", "t_start", "t_end"])
    .join(charts, on="stay_id", how="left")
    .filter(
        (pl.col("charttime") >= pl.col("t_start")) &
        (pl.col("charttime") <= pl.col("t_end"))
    )
)

# GCS (suma de componentes por stay, luego media en ventana)
gcs_all_ids = CHART_ITEMS["gcs_eye"] + CHART_ITEMS["gcs_verbal"] + CHART_ITEMS["gcs_motor"]
gcs_agg = (
    charts_win.filter(pl.col("itemid").is_in(gcs_all_ids))
    .group_by(["stay_id", "charttime"])
    .agg(pl.col("valuenum").sum().alias("gcs_total"))
    .group_by("stay_id")
    .agg(pl.col("gcs_total").mean())
)

# MAP (art preferido sobre no-invasivo)
map_art_agg = (
    charts_win.filter(pl.col("itemid").is_in(CHART_ITEMS["map_art"]))
    .group_by("stay_id")
    .agg(pl.col("valuenum").mean().alias("map_art"))
)
map_ni_agg = (
    charts_win.filter(pl.col("itemid").is_in(CHART_ITEMS["map_ni"]))
    .group_by("stay_id")
    .agg(pl.col("valuenum").mean().alias("map_ni"))
)

# SpO2 y FiO2
spo2_agg = (
    charts_win.filter(pl.col("itemid").is_in(CHART_ITEMS["spo2"]))
    .group_by("stay_id")
    .agg(pl.col("valuenum").mean().alias("spo2"))
)
fio2_agg = (
    charts_win.filter(pl.col("itemid").is_in(CHART_ITEMS["fio2"]))
    .group_by("stay_id")
    .agg(pl.col("valuenum").mean().alias("fio2"))
)

# Combinar
sofa_charts = (
    stays_win.select("stay_id")
    .join(gcs_agg, on="stay_id", how="left")
    .join(map_art_agg, on="stay_id", how="left")
    .join(map_ni_agg, on="stay_id", how="left")
    .join(spo2_agg, on="stay_id", how="left")
    .join(fio2_agg, on="stay_id", how="left")
)

# MAP: usar arterial, si no existe usar no-invasivo
sofa_charts = sofa_charts.with_columns(
    pl.when(pl.col("map_art").is_not_null())
    .then(pl.col("map_art"))
    .otherwise(pl.col("map_ni"))
    .alias("map_val")
)

# Aplicar puntuación SOFA charts
sofa_charts = sofa_charts.with_columns([
    # CNS (GCS)
    pl.when(pl.col("gcs_total").is_null()).then(0)
      .when(pl.col("gcs_total") >= 15).then(0)
      .when(pl.col("gcs_total") >= 13).then(1)
      .when(pl.col("gcs_total") >= 10).then(2)
      .when(pl.col("gcs_total") >= 6).then(3)
      .otherwise(4).cast(pl.Int8).alias("sofa_cns"),
    # Cardiovascular (MAP, sin vasopresores por ahora)
    pl.when(pl.col("map_val").is_null()).then(0)
      .when(pl.col("map_val") >= 70).then(0)
      .otherwise(1).cast(pl.Int8).alias("sofa_cardio_map"),
    # Respiratorio (SpO2/FiO2 surrogate)
    pl.when(pl.col("spo2").is_null() | pl.col("fio2").is_null()).then(0)
      .otherwise(
          pl.when(pl.col("spo2") / (pl.col("fio2").clip(21, 100) / 100) >= 400).then(0)
          .when(pl.col("spo2") / (pl.col("fio2").clip(21, 100) / 100) >= 300).then(1)
          .when(pl.col("spo2") / (pl.col("fio2").clip(21, 100) / 100) >= 235).then(2)
          .when(pl.col("spo2") / (pl.col("fio2").clip(21, 100) / 100) >= 148).then(3)
          .otherwise(4)
      ).cast(pl.Int8).alias("sofa_resp"),
])

print(f"SOFA charts calculado para {len(sofa_charts):,} stays")
sofa_charts.select(["stay_id", "gcs_total", "sofa_cns", "map_val", "sofa_cardio_map", "spo2", "fio2", "sofa_resp"]).head(5)

[19:35:05] Calculando SOFA charts (vectorizado) ...


SOFA charts calculado para 23,095 stays


[Vista previa de registros individuales omitida — DUA de PhysioNet: los datos de MIMIC-IV no se redistribuyen]


In [12]:
# ── Vasopresores vectorizado ─────────────────────────────────────
log("Calculando SOFA cardiovascular con vasopresores ...")

# Para cada stay, ¿hay vasopresores activos en la ventana?
vaso_win = (
    stays_win.select(["stay_id", "t_start", "t_end"])
    .join(vasopress, on="stay_id", how="left")
    .filter(
        (pl.col("starttime") <= pl.col("t_end")) &
        (pl.col("endtime")   >= pl.col("t_start"))
    )
    .group_by("stay_id")
    .agg(pl.col("itemid").count().alias("n_vaso"))
    .with_columns(
        (pl.col("n_vaso") > 0).alias("has_vasopressor")
    )
)

# Actualizar SOFA cardiovascular: vasopressor = SOFA 2+ (sobre MAP)
sofa_charts = (
    sofa_charts
    .join(vaso_win.select(["stay_id", "has_vasopressor"]), on="stay_id", how="left")
    .with_columns(
        pl.col("has_vasopressor").fill_null(False)
    )
    .with_columns(
        pl.when(pl.col("has_vasopressor")).then(2)
          .otherwise(pl.col("sofa_cardio_map"))
          .cast(pl.Int8).alias("sofa_cardio")
    )
)

print(f"Stays con vasopresores en ventana: {vaso_win['stay_id'].n_unique():,}")

[19:35:06] Calculando SOFA cardiovascular con vasopresores ...
Stays con vasopresores en ventana: 5,008


In [13]:
# ── SOFA total ───────────────────────────────────────────────────
sofa_total = (
    stays_win.select(["stay_id", "hadm_id"])
    .join(sofa_labs.select(["hadm_id", "sofa_coag", "sofa_liver", "sofa_renal"]), on="hadm_id", how="left")
    .join(sofa_charts.select(["stay_id", "sofa_cns", "sofa_cardio", "sofa_resp"]), on="stay_id", how="left")
    .with_columns([
        pl.col(c).fill_null(0) for c in ["sofa_coag", "sofa_liver", "sofa_renal", "sofa_cns", "sofa_cardio", "sofa_resp"]
    ])
    .with_columns(
        (pl.col("sofa_coag") + pl.col("sofa_liver") + pl.col("sofa_renal") +
         pl.col("sofa_cns")  + pl.col("sofa_cardio") + pl.col("sofa_resp")
        ).alias("sofa")
    )
)

print(f"SOFA calculado para {len(sofa_total):,} stays")
print("\nDistribución SOFA:")
print(sofa_total["sofa"].describe())
sofa_total.head(5)

SOFA calculado para 23,095 stays

Distribución SOFA:
shape: (9, 2)
┌────────────┬──────────┐
│ statistic  ┆ value    │
│ ---        ┆ ---      │
│ str        ┆ f64      │
╞════════════╪══════════╡
│ count      ┆ 23095.0  │
│ null_count ┆ 0.0      │
│ mean       ┆ 4.718857 │
│ std        ┆ 3.72365  │
│ min        ┆ 0.0      │
│ 25%        ┆ 1.0      │
│ 50%        ┆ 4.0      │
│ 75%        ┆ 7.0      │
│ max        ┆ 19.0     │
└────────────┴──────────┘


[Vista previa de registros individuales omitida — DUA de PhysioNet: los datos de MIMIC-IV no se redistribuyen]


## PASO 5 — Etiqueta Sepsis-3

In [14]:
cohort = (
    stays_inf
    .join(
        sofa_total.select(["stay_id", "sofa", "sofa_coag", "sofa_liver", "sofa_renal",
                           "sofa_cns", "sofa_cardio", "sofa_resp"]),
        on="stay_id",
        how="left",
    )
    .with_columns([
        pl.when(
            pl.col("infection_time").is_not_null() & (pl.col("sofa").fill_null(0) >= 2)
        ).then(1).otherwise(0).cast(pl.Int8).alias("sepsis"),
        pl.when(
            pl.col("infection_time").is_not_null() & (pl.col("sofa").fill_null(0) >= 2)
        ).then(pl.col("infection_time")).otherwise(None).alias("onset_time"),
    ])
    .unique(subset=["stay_id"])   # guardia: elimina duplicados residuales si los hubiera
)

n_sep   = cohort.filter(pl.col("sepsis") == 1)["stay_id"].n_unique()
n_no    = cohort.filter(pl.col("sepsis") == 0)["stay_id"].n_unique()
n_total = n_sep + n_no
prev    = 100 * n_sep / n_total

print("=" * 50)
print("  COHORTE FINAL")
print("=" * 50)
print(f"  Total stays únicos:   {n_total:,}  (debe ser ~74,829)")
print(f"  Sepsis (label=1):     {n_sep:,}  ({prev:.1f}%)")
print(f"  No sepsis (label=0):  {n_no:,}  ({100-prev:.1f}%)")
print(f"  Pacientes únicos:     {cohort['subject_id'].n_unique():,}")

  COHORTE FINAL
  Total stays únicos:   74,829  (debe ser ~74,829)
  Sepsis (label=1):     17,189  (23.0%)
  No sepsis (label=0):  57,640  (77.0%)
  Pacientes únicos:     54,551


In [15]:
# Distribución SOFA en casos de sepsis
print("Distribución SOFA en stays con infección sospechada:")
print(
    sofa_total["sofa"].value_counts().sort("sofa").head(15)
)

print("\nComponentes SOFA (media en casos con infección):")
print(
    sofa_total.select(["sofa_coag", "sofa_liver", "sofa_renal",
                       "sofa_cns", "sofa_cardio", "sofa_resp"]).mean()
)

Distribución SOFA en stays con infección sospechada:
shape: (15, 2)
┌──────┬───────┐
│ sofa ┆ count │
│ ---  ┆ ---   │
│ i8   ┆ u32   │
╞══════╪═══════╡
│ 0    ┆ 3312  │
│ 1    ┆ 2594  │
│ 2    ┆ 2196  │
│ 3    ┆ 1846  │
│ 4    ┆ 1996  │
│ …    ┆ …     │
│ 10   ┆ 921   │
│ 11   ┆ 705   │
│ 12   ┆ 439   │
│ 13   ┆ 320   │
│ 14   ┆ 141   │
└──────┴───────┘

Componentes SOFA (media en casos con infección):
shape: (1, 6)
┌───────────┬────────────┬────────────┬──────────┬─────────────┬───────────┐
│ sofa_coag ┆ sofa_liver ┆ sofa_renal ┆ sofa_cns ┆ sofa_cardio ┆ sofa_resp │
│ ---       ┆ ---        ┆ ---        ┆ ---      ┆ ---         ┆ ---       │
│ f64       ┆ f64        ┆ f64        ┆ f64      ┆ f64         ┆ f64       │
╞═══════════╪════════════╪════════════╪══════════╪═════════════╪═══════════╡
│ 0.546006  ┆ 0.420394   ┆ 0.778783   ┆ 1.25144  ┆ 0.513315    ┆ 1.20892   │
└───────────┴────────────┴────────────┴──────────┴─────────────┴───────────┘


## PASO 6 — Guardar cohorte

In [16]:
cohort_out = cohort.select([
    "subject_id", "hadm_id", "stay_id",
    "intime", "outtime", "los_hours",
    "sepsis", "onset_time", "infection_time",
    "sofa", "sofa_coag", "sofa_liver", "sofa_renal",
    "sofa_cns", "sofa_cardio", "sofa_resp",
])

output_path = OUT_DIR / "cohort.parquet"
cohort_out.write_parquet(output_path)
log(f"Guardado en: {output_path}")

summary = {
    "total_stays":     n_total,       # FIX: n_sep + n_no, no len(cohort)
    "sepsis_stays":    n_sep,
    "no_sepsis_stays": n_no,
    "prevalence_pct":  round(prev, 2),
    "unique_patients": cohort["subject_id"].n_unique(),
}
with open(OUT_DIR / "cohort_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
log("COMPLETADO.")

[19:35:06] Guardado en: ../data/processed/cohort.parquet
{
  "total_stays": 74829,
  "sepsis_stays": 17189,
  "no_sepsis_stays": 57640,
  "prevalence_pct": 22.97,
  "unique_patients": 54551
}
[19:35:06] COMPLETADO.
